# Validação de idempotência do pipeline (Case RJ - Grupo 2)

Este notebook registra a verificação feita em 22/10/2026 depois de executar o Job `visagio - job` (Landing_To_Bronze → Bronze_To_Silver → Silver_To_Gold → Gold_To_CSV) mais de uma vez sobre os mesmos CSVs de entrada.

**Critério do enunciado:** reexecutar o pipeline sobre os mesmos dados deve produzir o mesmo resultado, sem duplicar nem alterar indevidamente registros.

**Método:** (1) foto de todas as tabelas `*_raw`, `*_silver` e `*_gold` (linhas e checksum `xxhash64`) antes e depois de uma nova execução; (2) para as tabelas que divergirem, comparação linha a linha entre versões Delta consecutivas; (3) comparação do `_manifesto.csv` da exportação entre duas execuções.

Na foto (etapa 1) só `_ingestion_ts` (Bronze) fica fora do checksum. Por isso `canais_silver` e `chamados_hora_silver` aparecem como divergentes na etapa 2: cada uma tem uma coluna de linhagem que muda por definição a cada execução, `_treatment_ts` (`current_timestamp()` do tratamento) e `versao_bronze` (versão Delta da Bronze). A etapa 4 comprova que, sem essas colunas, o conteúdo é idêntico.

## 1. Foto das tabelas

Executada uma vez por rodada (`RODADA` = 1 antes e 2 depois de uma nova execução do Job). Nesta versão, o conjunto `META` exclui só `_ingestion_ts`; por isso as duas colunas de linhagem de Silver aparecem na etapa 2 como divergência de checksum.

In [0]:
from pyspark.sql import functions as F

CAT = "dados_vcredit.default"
RODADA = 2   # 1 na primeira foto, 2 na segunda
META = {"_ingestion_ts"}

tabelas = sorted(
    r.tableName for r in spark.sql(f"SHOW TABLES IN {CAT}").collect()
    if r.tableName.endswith(("_raw", "_silver", "_gold"))
)
linhas = []
for t in tabelas:
    df = spark.table(f"{CAT}.{t}")
    cols = [F.col(c) for c in df.columns if c not in META]
    r = df.agg(
        F.count(F.lit(1)).alias("n"),
        F.sum(F.xxhash64(*cols).cast("decimal(38,0)")).alias("ck"),
    ).first()
    linhas.append((t, r["n"], str(r["ck"])))

foto = spark.createDataFrame(linhas, "tabela string, linhas long, checksum string")
foto.write.mode("overwrite").saveAsTable(f"{CAT}.idempotencia_rodada_{RODADA}")
display(foto)

tabela,linhas,checksum
atendentes_desempenho_gold,25,8522453941924241774
atendentes_silver,25,-12967259087165184096
base_atendentes_raw,25,-32344595793491449292
base_motivos_raw,14,-7957994840943796617
base_silver,131838,-4156569145427647318575
canais_depara_silver,7,-11698609515089386191
canais_raw,4,-12912062611026271796
canais_silver,4,28251136641991680596
canal_escalonamento_gold,97,79398483948032894537
canal_faixa_espera_gold,13,-14632145875435094082


## 2. Tabelas com checksum diferente entre as rodadas

Resultado: apenas `canais_silver` e `chamados_hora_silver` divergiram, com o mesmo número de linhas. As demais tabelas raw, silver e gold tiveram linhas e checksum idênticos.

In [0]:
%sql
SELECT a.tabela, a.linhas, b.linhas AS linhas_2,
       a.checksum = b.checksum AS igual
FROM dados_vcredit.default.idempotencia_rodada_1 a
JOIN dados_vcredit.default.idempotencia_rodada_2 b USING (tabela)
WHERE a.checksum <> b.checksum OR a.linhas <> b.linhas

tabela,linhas,linhas_2,igual
canais_silver,4,4,false
chamados_hora_silver,131838,131838,false


## 3. Histórico Delta das duas tabelas

In [0]:
# Mostra só versão, horário e operação (sem userName/userId, que trazem dado pessoal do autor)
for t in ['chamados_hora_silver', 'canais_silver']:
    print(t)
    display(spark.sql(f"DESCRIBE HISTORY dados_vcredit.default.{t}")
            .select('version', 'timestamp', 'operation').limit(3))


version,timestamp,operation
7,2026-10-22T04:42:26.000Z,CREATE OR REPLACE TABLE AS SELECT
6,2026-10-22T03:01:44.000Z,CREATE OR REPLACE TABLE AS SELECT
5,2026-10-22T02:01:51.000Z,CREATE OR REPLACE TABLE AS SELECT


## 4. Comparação linha a linha entre versões Delta consecutivas

Sem as colunas de linhagem (`versao_bronze` e `_treatment_ts`), as duas versões mais recentes têm conteúdo idêntico (`EXCEPT ALL` nos dois sentidos retornou 0). A divergência de checksum veio só dessas colunas de metadado.

In [0]:
from pyspark.sql import functions as F

def comparar(tabela, coluna_metadado):
    t = f"dados_vcredit.default.{tabela}"
    hist = (spark.sql(f"DESCRIBE HISTORY {t}")
            .orderBy(F.desc("version")).limit(2).collect())
    atual, anterior = hist[0]["version"], hist[1]["version"]
    a = spark.sql(f"SELECT * EXCEPT ({coluna_metadado}) FROM {t} VERSION AS OF {atual}")
    b = spark.sql(f"SELECT * EXCEPT ({coluna_metadado}) FROM {t} VERSION AS OF {anterior}")
    so_anterior = b.exceptAll(a).count()
    so_atual = a.exceptAll(b).count()
    print(f"{tabela}: versão {anterior} -> {atual} | só na anterior: {so_anterior} | só na atual: {so_atual}")

comparar("chamados_hora_silver", "versao_bronze")
comparar("canais_silver", "_treatment_ts")

chamados_hora_silver: versão 6 -> 7 | só na anterior: 0 | só na atual: 0
canais_silver: versão 6 -> 7 | só na anterior: 0 | só na atual: 0
